# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/PillalamarriSrikanth/flyrank-/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

Unit of Analysis: One row represents one unique content asset identifier (content_hash_id) on a specific reporting day.

Time Window: A mid-panel snapshot isolating performance data for March 2026 (month=2026-03)

In [4]:
import os
from google.colab import userdata
import pandas as pd
import duckdb

# 1. Retrieve your Hugging Face READ token from Secrets
hf_token = userdata.get('HF_TOKEN')

# 2. Initialize DuckDB connection and configure Hugging Face authentication
con = duckdb.connect()
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{hf_token}')")

print("Fetching March 2026 partition slice via DuckDB...")

# 3. Directly target the month=2026-03 folder partitions
query = """
    SELECT *
    FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet')
    LIMIT 30000
"""

# 4. Execute and convert directly to a pandas DataFrame
df = con.sql(query).df()
print(f"Loaded snapshot of {len(df):,} rows.")

# 5. Verify Grain and Window Boundary
# The 'content_hash_id' column appears to be the unique identifier for content items.
grain_check = df['content_hash_id'].is_unique
print(f"Is every row a unique content item? {grain_check}")
# The 'report_date' column indicates the date for each record.
print(f"Date ranges seen: {df['report_date'].min()} to {df['report_date'].max()}")

Fetching March 2026 partition slice via DuckDB...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Loaded snapshot of 30,000 rows.
Is every row a unique content item? True
Date ranges seen: 2026-03-01 00:00:00 to 2026-03-01 00:00:00


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

Context/Identifiers: month, report_date, content_hash_id.

Available Features: gsc_clicks, gsc_impressions, gsc_avg_position, ctr_current.

Target/Label: label (engineered here as a proxy for high-volume traffic classification).

Excluded Fields: Future business metrics or downstream conversions are excluded to prevent data leakage.

In [5]:
# Check column availability and types across our categories
print("--- TARGET AND IDENTIFIER BUCKETS ---")
print("Context/Identifiers:", [col for col in ['url_id', 'content_id', 'month', 'date'] if col in df.columns])
print("Available Features:", [col for col in ['clicks_90d', 'impressions_90d', 'position_avg', 'position_shift'] if col in df.columns])

target_col = 'is_declining' if 'is_declining' in df.columns else 'label'
print(f"Target/Label Column identified: '{target_col}'")

# Check for missing values in the core feature space
print("\nMissing values per column:")
print(df[[col for col in ['clicks_90d', 'impressions_90d', 'position_avg', target_col] if col in df.columns]].isnull().sum())

--- TARGET AND IDENTIFIER BUCKETS ---
Context/Identifiers: ['month']
Available Features: []
Target/Label Column identified: 'label'

Missing values per column:
Series([], dtype: float64)


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

Context/Identifiers: month, report_date, content_hash_id.

Available Features: gsc_clicks, gsc_impressions, gsc_avg_position, ctr_current.

Target/Label: label (engineered here as a proxy for high-volume traffic classification).

Excluded Fields: Future business metrics or downstream conversions are excluded to prevent data leakage.

In [10]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split

# --- FACT PROOF QUERIES ---
print("Query 1 (Is grain unique?):", df['content_hash_id'].is_unique)
print(f"Query 2 (Scale): Contains {len(df):,} rows spanning {df['month'].iloc[0]}")

# Create a dummy label since 'is_declining' and 'label' columns are not present
# Assuming a classification task, we can binarize a relevant metric like 'gsc_clicks'
# For example, let's define 'is_high_clicks' as our target.
y_label = (df['gsc_clicks'] > df['gsc_clicks'].median()).astype(int)
print(f"Query 3 (Availability): {y_label.sum():,} rows are flagged as high clicks (IS TRUE)\n")

# --- FEATURE ENGINEERING (5 FEATURES) ---
X_clean = pd.DataFrame()
# Use available columns from df for features
X_clean['gsc_clicks'] = df['gsc_clicks'] # using gsc_clicks as feature
X_clean['gsc_impressions'] = df['gsc_impressions'] # using gsc_impressions as feature
X_clean['gsc_avg_position'] = df['gsc_avg_position'] # using gsc_avg_position as feature
X_clean['ctr_current'] = df['gsc_clicks'] / (df['gsc_impressions'] + 1) # derived from current clicks/impressions
# 'position_shift' is not available in the current df, setting to 0 as a placeholder
X_clean['position_shift_1w'] = 0 # Placeholder as 'position_shift' is not in df

# --- LEAKAGE TRAP EXPERIMENT ---
X_leaked = X_clean.copy()
X_leaked['leaked_future_metric'] = y_label * 0.95 + 0.05  # Cheating target dependency injected here

# Train clean model
X_tr_c, X_te_c, y_tr, y_te = train_test_split(X_clean, y_label, test_size=0.3, random_state=42)
clf_c = RandomForestClassifier(max_depth=3, random_state=42).fit(X_tr_c, y_tr)
auc_clean = roc_auc_score(y_te, clf_c.predict_proba(X_te_c)[:, 1])

# Train leaked model
X_tr_l, X_te_l, _, _ = train_test_split(X_leaked, y_label, test_size=0.3, random_state=42)
clf_l = RandomForestClassifier(max_depth=3, random_state=42).fit(X_tr_l, y_tr)
auc_leaked = roc_auc_score(y_te, clf_l.predict_proba(X_te_l)[:, 1])

print("--- LEAKAGE EXPERIMENT RESULTS ---")
print(f"Clean Model ROC-AUC: {auc_clean:.4f}")
print(f"Leaked Model ROC-AUC: {auc_leaked:.4f} ⚠️ (Cheat Score!)")

# Drop the trap column to safeguard production dataset
print("\n[Action] Purged 'leaked_future_metric' column to secure dataset safety.")
X_final = X_clean

Query 1 (Is grain unique?): True
Query 2 (Scale): Contains 30,000 rows spanning 2026-03
Query 3 (Availability): 1,045 rows are flagged as high clicks (IS TRUE)

--- LEAKAGE EXPERIMENT RESULTS ---
Clean Model ROC-AUC: 1.0000
Leaked Model ROC-AUC: 1.0000 ⚠️ (Cheat Score!)

[Action] Purged 'leaked_future_metric' column to secure dataset safety.


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

This static snapshot cannot account for broader macroeconomic seasonality, site-wide platform tracking dropouts, or external Google search core algorithm updates that mimic organic content decay trends.

In [13]:
# Show the distribution imbalance of our dataset target flag
print("--- TARGET BASELINE DISTRIBUTION ---")
print(y_label.value_counts(normalize=True) * 100)
print("\nSummary statistics of raw search impressions:")
print(df['gsc_impressions'].describe())

--- TARGET BASELINE DISTRIBUTION ---
gsc_clicks
0    96.516667
1     3.483333
Name: proportion, dtype: float64

Summary statistics of raw search impressions:
count    30000.000000
mean        23.505333
std        122.651225
min          0.000000
25%          0.000000
50%          0.000000
75%          4.000000
max       6912.000000
Name: gsc_impressions, dtype: float64


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.